In [1]:
import pandas as pd
import numpy as np

In [14]:
tournament_df = pd.read_csv("../tournament.csv")

cols = ["match_id", "tournament_id", "tournament_name",
        "tournament_category_name", "ground_type"]
tournament_df = tournament_df[cols].copy()


In [15]:
# ─────────────────────────────────────────────
# 2. DROP ROWS MISSING tournament_id
# ─────────────────────────────────────────────
before = len(tournament_df)
tournament_df = tournament_df.dropna(subset=["tournament_id"])
print(f"[Info] Dropped {before - len(tournament_df):,} rows missing tournament_id")

tournament_df["tournament_id"] = pd.to_numeric(
    tournament_df["tournament_id"], errors="coerce"
).astype("Int64")
tournament_df = tournament_df.dropna(subset=["tournament_id"])

[Info] Dropped 0 rows missing tournament_id


In [16]:
# ─────────────────────────────────────────────
# 3. HANDLE MISSING ground_type
# Raw data has NaN rows (e.g. Davis Cup) — fill as "Unknown"
# ─────────────────────────────────────────────
missing_surface = tournament_df["ground_type"].isna().sum()
print(f"[Info] {missing_surface:,} rows with missing ground_type → filled as 'Unknown'")
tournament_df["ground_type"] = tournament_df["ground_type"].fillna("Unknown")



[Info] 562 rows with missing ground_type → filled as 'Unknown'


In [17]:
# ─────────────────────────────────────────────
# 4. VALIDATE AGAINST KNOWN VALUES
# Raw data is already clean — just confirm no surprises
# ─────────────────────────────────────────────
known_surfaces = {"Hard", "Clay", "Hard (Indoor)", "Grass", "Carpet", "Unknown"}

unexpected = tournament_df[~tournament_df["ground_type"].isin(known_surfaces)]
if not unexpected.empty:
    print(f"[Warning] {len(unexpected):,} rows with unexpected surface values:")
    print(unexpected["ground_type"].value_counts().to_string())
    # Strip whitespace in case of invisible padding
    tournament_df["ground_type"] = tournament_df["ground_type"].str.strip()
else:
    print("[Info] All ground_type values are recognised — no mapping needed")

# ─────────────────────────────────────────────
# 5. DROP EXACT DUPLICATE ROWS
# ─────────────────────────────────────────────
tournament_df = tournament_df.drop_duplicates()


[Warning] 34,600 rows with unexpected surface values:
ground_type
Hardcourt outdoor    16959
Red clay             11856
Hardcourt indoor      4741
Red clay indoor        512
Carpet indoor          276
Synthetic outdoor      226
Green clay              30


In [18]:
# ─────────────────────────────────────────────
# 6. RESOLVE CONFLICTING SURFACE PER TOURNAMENT
# Same tournament_id with different ground_type → take MODE
# ─────────────────────────────────────────────
def resolve_mode(series):
    mode = series.mode()
    if len(mode) == 1:
        return mode.iloc[0]
    non_unknown = [m for m in mode if m != "Unknown"]
    return non_unknown[0] if non_unknown else mode.iloc[0]

tournament_unique = (
    tournament_df
    .groupby("tournament_id")
    .agg(
        ground_type              = ("ground_type",              resolve_mode),
        tournament_name          = ("tournament_name",          "first"),
        tournament_category_name = ("tournament_category_name", "first"),
    )
    .reset_index()
)

print(f"\nUnique tournaments after cleaning: {len(tournament_unique):,}\n")



Unique tournaments after cleaning: 535



In [19]:
# ─────────────────────────────────────────────
# 7. SEPARATE UNKNOWN FROM KNOWN
# ─────────────────────────────────────────────
unknown_count = (tournament_unique["ground_type"] == "Unknown").sum()
known_df = tournament_unique[tournament_unique["ground_type"] != "Unknown"].copy()
print(f"[Info] {unknown_count} tournaments excluded (unknown surface)\n")


[Info] 6 tournaments excluded (unknown surface)



In [20]:
# ─────────────────────────────────────────────
# 8. SURFACE DISTRIBUTION BY UNIQUE TOURNAMENT
# ─────────────────────────────────────────────
distribution = (
    known_df["ground_type"]
    .value_counts()
    .reset_index()
)
distribution.columns = ["Surface", "Tournament Count"]
distribution["Percentage (%)"] = (
    (distribution["Tournament Count"] /
     distribution["Tournament Count"].sum() * 100)
    .round(2)
)

print("=" * 50)
print("  Most Common Tournament Surfaces")
print("=" * 50)
print(distribution.to_string(index=False))
print()

  Most Common Tournament Surfaces
          Surface  Tournament Count  Percentage (%)
Hardcourt outdoor               237           44.80
         Red clay               181           34.22
 Hardcourt indoor                85           16.07
  Red clay indoor                 8            1.51
            Grass                 8            1.51
    Carpet indoor                 5            0.95
Synthetic outdoor                 4            0.76
       Green clay                 1            0.19



In [21]:


# ─────────────────────────────────────────────
# 9. DISTRIBUTION BY MATCH COUNT
# ─────────────────────────────────────────────
match_surface = (
    tournament_df[["match_id", "tournament_id"]]
    .drop_duplicates()
    .merge(tournament_unique[["tournament_id", "ground_type"]], on="tournament_id")
)
match_surface = match_surface[match_surface["ground_type"] != "Unknown"]

match_dist = (
    match_surface["ground_type"]
    .value_counts()
    .reset_index()
)
match_dist.columns = ["Surface", "Match Count"]
match_dist["Percentage (%)"] = (
    (match_dist["Match Count"] /
     match_dist["Match Count"].sum() * 100)
    .round(2)
)

print("=" * 50)
print("  Surface Distribution by Match Count")
print("=" * 50)
print(match_dist.to_string(index=False))
print()


  Surface Distribution by Match Count
          Surface  Match Count  Percentage (%)
Hardcourt outdoor         8093           48.68
         Red clay         5583           33.58
 Hardcourt indoor         2188           13.16
            Grass          267            1.61
  Red clay indoor          236            1.42
    Carpet indoor          126            0.76
Synthetic outdoor          114            0.69
       Green clay           18            0.11



In [22]:
# ─────────────────────────────────────────────
# 10. SURFACE BY TOUR (ATP / ITF / WTA etc.)
# ─────────────────────────────────────────────
tour_surface = (
    known_df
    .groupby(["tournament_category_name", "ground_type"])
    .size()
    .reset_index(name="Count")
    .sort_values(["tournament_category_name", "Count"], ascending=[True, False])
)
tour_surface.columns = ["Tour", "Surface", "Tournament Count"]

print("=" * 55)
print("  Surface Distribution by Tour")
print("=" * 55)
print(tour_surface.to_string(index=False))


  Surface Distribution by Tour
            Tour           Surface  Tournament Count
             ATP Hardcourt outdoor                14
             ATP          Red clay                13
             ATP  Hardcourt indoor                 7
      Challenger          Red clay                34
      Challenger Hardcourt outdoor                20
      Challenger  Hardcourt indoor                17
Challenger Women Hardcourt outdoor                 6
Challenger Women          Red clay                 6
         ITF Men Hardcourt outdoor                95
         ITF Men          Red clay                70
         ITF Men  Hardcourt indoor                23
         ITF Men             Grass                 4
         ITF Men   Red clay indoor                 4
         ITF Men     Carpet indoor                 3
       ITF Women Hardcourt outdoor                87
       ITF Women          Red clay                57
       ITF Women  Hardcourt indoor                35
       ITF Wome